In [59]:
import math
import random

In [60]:
class Value():
  def __init__(self,data,_children=(), _op=''):
    self.data = data
    self._prev = set(_children)
    self.grad = 0.0
    self._backward = lambda : None
    self._op = _op

  def __add__ (self,other):
    other = other if isinstance (other,Value) else Value(other)
    out = Value(self.data + other.data, (self,other), '+')
    def _backward():
      self.grad += out.grad
      other.grad += out.grad
    out._backward = _backward
    return out

  def __mul__ (self,other):
    other = other if isinstance (other,Value) else Value(other)
    out = Value(self.data * other.data, (self,other), '*')
    def _backward():
      self.grad += out.grad * other.data
      other.grad += out.grad * self.data
    out._backward = _backward
    return out

  def __repr__(self):
    return f"Value(data={self.data}, grad={self.grad})"

  def __truediv__ (self,other):
    return self * other**-1

  def __pow__ (self,other):
    assert isinstance (other, (int, float))
    out = Value(self.data**other, (self,), f"**{other}")

    def _backward():
      self.grad += (other * self.data**(other-1)) * out.grad
    out._backward = _backward
    return out

  def tanh (self) :
    x = self.data
    t = (math.exp(x*2) - 1) / (math.exp(x*2) + 1)
    out = Value(t, (self,), 'tanh')
    def _backward():
      self.grad += (1 - out.data**2) * out.grad
    out._backward = _backward
    return out

  def __rmul__(self,other):
    return self * other
  def __radd__(self,other):
    return self + other
  def __neg__(self):
    return self * (-1)
  def __sub__(self, other):
    return self + (-other)

  def exp (self):
    x = self.data
    out = Value(math.exp(x), (self,), 'exp')
    def _backward():
      self.grad += out.data * out.grad ## local derivative * global derivative
    out._backward = _backward
    return out



 ##Görev 3 - Topolojik Liste
  def backward(self):
    topo = []
    visited = set()
    def build(v):
      if v not in visited:
         visited.add(v)
         for child in v._prev:
          build(child)
         topo.append(v)
    build(self)
    self.grad = 1.0
    for node in reversed(topo):
       node._backward()

In [61]:
##Numerical Derivative


def numeratical_derivative():
 h = 0.0001
 a = Value(2.0)
 b = Value(-3.0)
 c = Value(10.0)
 e = a*b
 d = e + c
 f = Value(-2.0)
 L = d * f
 L1 = L.data

 a = Value(2.0 + h) ## a değişkenini çok az arttırdık h kadar.
 b = Value(-3.0)
 c = Value(10.0)
 e = a*b
 d = e + c
 f = Value(-2.0)
 L = d * f
 L2 = L.data

 print((L2 - L1)/h)
numeratical_derivative()

6.000000000021544


In [62]:
## Analytic Derivative

def analytic_derivative():
 a = Value(2.0)
 b = Value(-3.0)
 c = Value(10.0)
 e = a*b
 d = e + c
 f = Value(-2.0)
 L = d * f
 L.backward()
 print(a.grad)
analytic_derivative()

6.0


In [63]:
## a'nın türevini manuel hesaplıyorum chain rule ile
def manuel_backprop():
  a = Value(2.0)
  b = Value(-3.0)
  c = Value(10.0)
  e = a*b
  d = e + c
  f = Value(-2.0)
  L = d * f
  L.backward()


  dL_dd = f.data # -2
  dd_de = 1 ## Toplama'da artış direkt geçer o yüzden türev 1, aynı oranda arttırır.
  de_da = b.data # -3

# by chain rule a.grad = 6
  dL_da = (dL_dd) * (dd_de) * (de_da)
  print(a.grad, dL_da)
manuel_backprop()

6.0 6.0


In [64]:
def gorev_2_manuel():
 x1 = Value(2.0)
 x2 = Value(0.0)

 w1 = Value(-3.0)
 w2 = Value(1.0)

 b = Value(6.8813735870195432)

 #forward pass
 x1w1 = x1*w1
 x2w2 = x2*w2
 x1w1x2w2 = x1w1 + x2w2
 n = x1w1x2w2 + b
 o = n.tanh()

 o.grad = 1.0 #Starting Point
 n.grad = o.grad * (1-o.data**2)
 b.grad = n.grad
 x1w1x2w2.grad = n.grad
 x1w1.grad = x1w1x2w2.grad
 w1.grad = x1w1.grad * x1.data
 x1.grad = x1w1.grad * w1.data
 x2w2.grad = x1w1x2w2.grad
 x2.grad = x2w2.grad * w2.data
 w2.grad = x2w2.grad * x2.data

 print(x1.grad, w1.grad, x2.grad, w2.grad, n.grad)

gorev_2_manuel()





-1.4999999999999996 0.9999999999999998 0.4999999999999999 0.0 0.4999999999999999


In [65]:
## Nöron'a tanh bağladım.

def gorev_2():
 x1 = Value(2.0)
 x2 = Value(0.0)

 w1 = Value(-3.0)
 w2 = Value(1.0)

 b = Value(6.8813735870195432)

 #forward pass
 x1w1 = x1*w1
 x2w2 = x2*w2
 x1w1x2w2 = x1w1 + x2w2
 n = x1w1x2w2 + b
 o = n.tanh()
 o.backward()



 print(o)

gorev_2()

Value(data=0.7071067811865476, grad=1.0)


In [78]:
#Analitik Derivative
def gorev_4_1_A():
    x1 = Value(2.0)
    x2 = Value(0.0)

    w1 = Value(-3.0)
    w2 = Value(1.0)

    b = Value(6.8813735870195432)


    x1w1 = x1*w1
    x2w2 = x2*w2
    x1w1x2w2 = x1w1 + x2w2
    n = x1w1x2w2 + b
    o = n.tanh()
    o.backward()
    print(x1.grad, w1.grad, x2.grad, w2.grad, n.grad)

gorev_4_1_A()



-1.4999999999999996 0.9999999999999998 0.4999999999999999 0.0 0.4999999999999999


In [67]:
## Numerical Derivative

def gorev_4_1():
  x1 = Value(2.0)
  x2 = Value(0.0)

  w1 = Value(-3.0)
  w2 = Value(1.0)

  b = Value(6.8813735870195432)

  h = 0.00001
  x1w1 = x1*w1
  x2w2 = x2*w2
  x1w1x2w2 = x1w1 + x2w2
  n = x1w1x2w2 + b
  o = n.tanh()
  o1 = o



  x1w1 = x1*(w1+h)  # do / dw1 = (do / dn) * (dn / d(x2w1)) * (d(x2w1) / w1) ## by chain rule
  x2w2 = x2*w2
  x1w1x2w2 = x1w1 + x2w2
  n = x1w1x2w2 + b
  o = n.tanh()
  o2 = o

  print((o2.data - o1.data) / h)



gorev_4_1()




0.9999858579412545


In [68]:
## Tanh parçalanmış şekilde

def gorev_4_2():
   x1 = Value(2.0)
   x2 = Value(0.0)

   w1 = Value(-3.0)
   w2 = Value(1.0)

   b = Value(6.8813735870195432)

 #forward pass
   x1w1 = x1*w1
   x2w2 = x2*w2
   x1w1x2w2 = x1w1 + x2w2
   n = x1w1x2w2 + b
   e = (n*2).exp()
   o = (e-1) / (e+1)
   o.backward()
   print(('x1=', x1.grad),('(w1=',w1.grad), ('x2=',x2.grad), ('w2=',w2.grad), ('n=', n.grad))

gorev_4_2()




('x1=', -1.5) ('(w1=', 1.0) ('x2=', 0.5) ('w2=', 0.0) ('n=', 0.5)


In [69]:

##Pytorch
def gorev_4_3():

    import torch

    x1 = torch.Tensor([2.0]).double()      ; x1.requires_grad = True
    x2 = torch.Tensor([0.0]).double()      ; x2.requires_grad = True
    w1 = torch.Tensor([-3.0]).double()     ; w1.requires_grad = True
    w2 = torch.Tensor([1.0]).double()      ; w2.requires_grad = True
    b  = torch.Tensor([6.8813735870195432]).double()         ; b.requires_grad  = True
    n = x1*w1 + x2*w2 + b
    n.retain_grad()
    o = torch.tanh(n)

    print(o.data.item())
    o.backward()


    print('---')
    print('x2=', x2.grad.item())
    print('w2=', w2.grad.item())
    print('x1=', x1.grad.item())
    print('w1=', w1.grad.item())
    print('n=', n.grad.item())

gorev_4_3()


0.7071066904050358
---
x2= 0.5000001283844369
w2= 0.0
x1= -1.5000003851533106
w1= 1.0000002567688737
n= 0.5000001283844369


In [70]:
class Neuron():
  def __init__ (self,nin):
    self.w = [Value(random.uniform(-1,1)) for _ in range (nin)]
    self.b = Value(random.uniform(-1,1))

  def __call__(self,x):
    act = sum((wi*xi for wi , xi in zip (self.w, x)) , self.b)
    out = act.tanh()
    return out

  def parameters(self):
    return self.w + [self.b]

class Layer():
  def __init__(self,nin,nout):
    self.neurons = [Neuron(nin) for _ in range(nout)]

  def __call__(self,x):
     outs = [n(x) for n in self.neurons]
     return outs[0] if len(outs) ==1 else outs

  def parameters(self):
    return [p for neuron in self.neurons for p in neuron.parameters()]

class MLP():
  def __init__ (self, nin, nouts):
    sz = [nin] + nouts
    self.layers = [Layer(sz[i] , sz[i+1]) for i in range(len(nouts))]
  def __call__ (self,x):
    for layer in self.layers:
      x = layer(x)
    return x

  def parameters(self):
    return [p for layer in self.layers for p in layer.parameters()]

x = [2.0, 3.0, -1.0]
n = MLP (3, [4,4,1])
n(x)

n.parameters()

[Value(data=-0.14133846425212093, grad=0.0),
 Value(data=-0.6850012734766342, grad=0.0),
 Value(data=-0.7028684768480113, grad=0.0),
 Value(data=-0.2962570228195447, grad=0.0),
 Value(data=0.9575154283883576, grad=0.0),
 Value(data=0.6535400820670738, grad=0.0),
 Value(data=0.5732500425292122, grad=0.0),
 Value(data=-0.38976377400029194, grad=0.0),
 Value(data=0.22808116810838097, grad=0.0),
 Value(data=-0.697179478745581, grad=0.0),
 Value(data=-0.4174366773356688, grad=0.0),
 Value(data=0.6637041148553127, grad=0.0),
 Value(data=-0.06250985433671508, grad=0.0),
 Value(data=0.6389216093909036, grad=0.0),
 Value(data=-0.5972666804391802, grad=0.0),
 Value(data=-0.24881502246470433, grad=0.0),
 Value(data=-0.5753864140761107, grad=0.0),
 Value(data=-0.042501873485738084, grad=0.0),
 Value(data=0.5712569096117701, grad=0.0),
 Value(data=-0.12762976339379617, grad=0.0),
 Value(data=0.05625648750889378, grad=0.0),
 Value(data=-0.701581325765537, grad=0.0),
 Value(data=-0.5369280865965513, 

In [71]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]  # desired targets

ypred = [n(x) for x in xs]
ypred

[Value(data=0.8394220404527581, grad=0.0),
 Value(data=0.2469761563375817, grad=0.0),
 Value(data=0.7973881061231809, grad=0.0),
 Value(data=0.8117142908927284, grad=0.0)]

In [72]:
loss = sum((yout - ygt)**2 for ygt , yout in zip(ys, ypred))
loss

Value(data=4.846790327853907, grad=0.0)

In [73]:
loss.backward()
n.parameters()


[Value(data=-0.14133846425212093, grad=-0.5968313358332241),
 Value(data=-0.6850012734766342, grad=0.2062373106381182),
 Value(data=-0.7028684768480113, grad=-0.02157235808908987),
 Value(data=-0.2962570228195447, grad=-0.1776553638791936),
 Value(data=0.9575154283883576, grad=0.1554561403192639),
 Value(data=0.6535400820670738, grad=-0.04797237555407121),
 Value(data=0.5732500425292122, grad=0.03620724822280721),
 Value(data=-0.38976377400029194, grad=0.05528771211468297),
 Value(data=0.22808116810838097, grad=-0.479422933789852),
 Value(data=-0.697179478745581, grad=-0.22671937316835467),
 Value(data=-0.4174366773356688, grad=-0.5513923448825776),
 Value(data=0.6637041148553127, grad=-0.4930414441194735),
 Value(data=-0.06250985433671508, grad=0.9976135993307728),
 Value(data=0.6389216093909036, grad=-0.08990870637922817),
 Value(data=-0.5972666804391802, grad=0.38741760716466),
 Value(data=-0.24881502246470433, grad=0.5101401779615583),
 Value(data=-0.5753864140761107, grad=1.029525

In [74]:
for k in range(50):

  ypred = [n(x) for x in xs]
  loss = sum((yout - ygt)**2 for ygt , yout in zip(ys, ypred))
  for p in n.parameters():
    p.grad = 0
  loss.backward()

  for p in n.parameters():
    p.data += -0.05 * p.grad

  print(k, loss.data)



0 4.846790327853907
1 2.9304050977670895
2 1.807431999377462
3 1.0873931356490811
4 0.5964685570211463
5 0.35227617527057464
6 0.23566078498980875
7 0.1729186262303668
8 0.13499245866634138
9 0.1099774097407305
10 0.09238810796221597
11 0.07941297663439291
12 0.06948170920291746
13 0.0616553062281069
14 0.055340631801674034
15 0.050145892818691216
16 0.045802532394344245
17 0.0421206525850969
18 0.03896237602606544
19 0.03622529088330167
20 0.03383180881390147
21 0.03172212000431966
22 0.029849407718009858
23 0.028176522793286404
24 0.026673625381569095
25 0.025316481978721234
26 0.02408521540639099
27 0.022963373602087514
28 0.02193722651925053
29 0.020995228702773525
30 0.020127603852752425
31 0.019326020345461846
32 0.01858333536423349
33 0.017893391340479264
34 0.01725085267525956
35 0.016651073765690895
36 0.016089991570629767
37 0.01556403756736141
38 0.015070065146776664
39 0.014605289387203605
40 0.014167236819550224
41 0.013753703307370392
42 0.013362718556808172
43 0.01299251

In [75]:
ypred

[Value(data=0.957933701535466, grad=-0.08413259692906805),
 Value(data=-0.9469542139238661, grad=0.10609157215226772),
 Value(data=-0.9349821579581675, grad=0.13003568408366495),
 Value(data=0.9518542569610432, grad=-0.0962914860779136)]